# Edge dataset audit

50 stratified samples of the sim table with their images.


In [ ]:
import glob, json, os, pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 250)
tab = glob.glob("/kaggle/input/**/simtable_v1.jsonl", recursive=True)[0]
idx = glob.glob("/kaggle/input/**/scheduler-dataset-v1-images/**/index.jsonl", recursive=True)[0]
root = os.path.dirname(idx)
df = pd.read_json(tab, lines=True)
files = {}
for line in open(idx):
    r = json.loads(line); files[r.get("uid") or r.get("image_id")] = r.get("file") or r.get("path")
T = ["Critical", "High", "Medium", "Low"]
M = {0: "general", 1: "flood_storm", 2: "wildfire", 3: "earthquake"}
df["true_tier"] = df["tier"].map(dict(enumerate(T)))
df["spi_top"] = df["tier_probs"].map(lambda p: T[max(range(4), key=lambda i: p[i])])
df["spi"] = df["tier_probs"].map(lambda p: [round(x, 2) for x in p])
df["mission_name"] = df["mission"].map(M)
df["sts_hist7"] = df["sts"].map(lambda s: [round(x, 2) for x in s[:7]])
print(len(df), "rows"); display(df.groupby(["split", "true_tier"]).size().unstack())

In [ ]:
# 50 samples: stratified by split x true tier (as balanced as the data allows), fixed seed
s = df.groupby(["split", "true_tier"], group_keys=False).apply(lambda g: g.sample(min(len(g), 5), random_state=7))
s = s.sample(min(50, len(s)), random_state=7).reset_index(drop=True)
cols = ["image_id", "split", "event", "source", "true_tier", "spi_top", "spi", "relevance", "mission_name",
        "payload_bytes", "sts_hist7", "dup_group", "capture_time_s"]
display(s[cols])

In [ ]:
fig, axes = plt.subplots(10, 5, figsize=(22, 46))
for ax, (_, r) in zip(axes.ravel(), s.iterrows()):
    f = files.get(r["image_id"])
    try:
        ax.imshow(Image.open(os.path.join(root, f)).convert("RGB"))
    except Exception as e:
        ax.text(0.5, 0.5, "missing: %s" % e, ha="center")
    ax.set_title("%s | true %s | SPI %s\nP=%s rel %.2f %s" % (r["split"], r["true_tier"], r["spi_top"], r["spi"],
                 r["relevance"], r["mission_name"]), fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/audit_grid.jpg", dpi=60); plt.show()
s[cols].to_csv("/kaggle/working/audit_50.csv", index=False)